In [1]:
!pip install -U langchain langchain-community langchain-google-genai langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 kB 11.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.8 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1

In [2]:
!pip install langchain_openai

In [1]:
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

In [ ]:
# Load environment variables
# load_dotenv()

# Initialize the chat model used to generate responses
# llm = ChatOpenAI()


In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

In [3]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",api_key=GOOGLE_API_KEY)

# Define the state schema shared across nodes in the graph
class ChatState(TypedDict):
    # `add_messages` appends new messages to the list rather than overwriting previous ones
    messages: Annotated[list[BaseMessage], add_messages]


# Node function that takes the current conversation state and calls the LLM
def chat_node(state: ChatState):
    messages = state["messages"]
    response = llm.invoke(messages)
    # Return only the state update; LangGraph merges it using the reducer (`add_messages`)
    return {"messages": [response]}


# In-memory checkpointer to persist conversation history per thread_id
checkpointer = InMemorySaver()

# Build the state graph with the defined state schema
graph = StateGraph(ChatState)

# Register the chat node and connect entry -> chat_node -> exit
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

# Compile the graph into an executable chatbot with memory enabled
chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
!pip install -q grandalf

In [4]:
# import of grandalf
import grandalf

In [5]:
# Print the ASCII representation of the state graph workflow
chatbot.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| chat_node |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [7]:
# Define input that matches ChatState schema and provide a thread_id config
input_state = {"messages": [("user", "What is the capital of India?")]}
config = {"configurable": {"thread_id": "conversation_thread_1"}}

# Invoke the chatbot with the state and config
chatbot.invoke(input_state, config)

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='155c72dc-68f7-4ccd-b6b9-e1860035735e'),
  AIMessage(content=[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EnsKeQFpFH0TJyyacVreAAubyNSyY1AB03fw0DSnuarnwQrTSeX5YdB3BBb7x7d5mTWfvaDvjKhAdkQr6AHxeQwjxanVK+dCfEffme5uEwvfM4McnfwzyXo7x5mdLFN+iS5PX1o7+6mo37fJX82Rblq5ydw8OSs+Mmoiy0E='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f549-4e79-72f5-90c5-db11abddbc1e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}})]}

In [9]:
from langchain_core.output_parsers import StrOutputParser

# Initialize the output parser
output_parser = StrOutputParser()

# 1. Invoke the chatbot to get the full state back
state_output = chatbot.invoke(input_state, config)

# 2. Extract the last AI message from the updated message list
last_message = state_output["messages"][-1]

# 3. Parse the message to clean text using the output parser
parsed_response = output_parser.invoke(last_message)
print(parsed_response)

The capital of India is New Delhi.


In [10]:
state_output

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='155c72dc-68f7-4ccd-b6b9-e1860035735e'),
  AIMessage(content=[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EnsKeQFpFH0TJyyacVreAAubyNSyY1AB03fw0DSnuarnwQrTSeX5YdB3BBb7x7d5mTWfvaDvjKhAdkQr6AHxeQwjxanVK+dCfEffme5uEwvfM4McnfwzyXo7x5mdLFN+iS5PX1o7+6mo37fJX82Rblq5ydw8OSs+Mmoiy0E='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f549-4e79-72f5-90c5-db11abddbc1e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='81dfaaac-40c6-4b74-9b8b-736a5041d10c'),
  AIMessage(content=[{'type': 'text', 'te

In [14]:
last_message

AIMessage(content=[{'type': 'text', 'text': 'The capital of India is New Delhi.', 'extras': {'signature': 'EnsKeQFpFH0TK7yoQ8TUVv0l5iN1Pos/KUui+lLw2LQ8bZvGqESUu3Qxb/o+GLmfLW/14R0INtu86obctMuB8LF+pljkmkC+e4INMEGlkQTsSp+U4iI160vwtM2Hk9DCMSuRJff1iic0mogOnCKwcnuGawVvNzPu3w52Ykw='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f54e-5cd1-742d-adfe-f180472e43da-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 8, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}})

In [15]:
from langchain_core.output_parsers import JsonOutputParser

# Initialize the output parser
output_parser = JsonOutputParser()

# 1. Invoke the chatbot to get the full state back
state_output = chatbot.invoke(input_state, config)

# 2. Extract the last AI message from the updated message list
last_message = state_output["messages"][-1]

In [18]:
output_parser

JsonOutputParser()

In [16]:
state_output

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='155c72dc-68f7-4ccd-b6b9-e1860035735e'),
  AIMessage(content=[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EnsKeQFpFH0TJyyacVreAAubyNSyY1AB03fw0DSnuarnwQrTSeX5YdB3BBb7x7d5mTWfvaDvjKhAdkQr6AHxeQwjxanVK+dCfEffme5uEwvfM4McnfwzyXo7x5mdLFN+iS5PX1o7+6mo37fJX82Rblq5ydw8OSs+Mmoiy0E='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f549-4e79-72f5-90c5-db11abddbc1e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='81dfaaac-40c6-4b74-9b8b-736a5041d10c'),
  AIMessage(content=[{'type': 'text', 'te

In [17]:
last_message

AIMessage(content=[{'type': 'text', 'text': 'The capital of India is New Delhi.', 'extras': {'signature': 'EnsKeQFpFH0TIjlxaKeSMUW6W613Dvm5xIo8rSRpudozo2+xYkQp+KcyDEJWvHnp48ZzjeefzHoxf69FQ2D5UMAcUBpxmeVHv+aIQ49YQP5AS7rfS7vBphVlmCXVVwgdVFumZX5LPY1FSsCOcXmnrz5HI7/8q4+or3uofuA='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1f550-e933-75a7-a34d-414b1dd5a589-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 8, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}})

In [20]:
from langchain_core.output_parsers import StrOutputParser

# Initialize the string output parser instead of JSON parser
str_parser = StrOutputParser()

# 3. Parse the message to clean text using the output parser
parsed_response = str_parser.invoke(last_message)
print(parsed_response)

The capital of India is New Delhi.


### Querying the Chatbot to Return Valid JSON

In [21]:
from langchain_core.output_parsers import JsonOutputParser

# 1. Initialize the JSON output parser
json_parser = JsonOutputParser()

# 2. Request the output in JSON format by giving clear prompt instructions
json_input_state = {
    "messages": [("user", "What is the capital of India? Respond ONLY with a JSON object containing the keys 'country' and 'capital'.") ]
}
json_config = {"configurable": { "thread_id": "json_conversation_thread" }}

# 3. Invoke the graph
json_state_output = chatbot.invoke(json_input_state, json_config)
json_last_message = json_state_output["messages"][-1]

# 4. Parse the output into a Python dictionary
parsed_json = json_parser.invoke(json_last_message)
print(type(parsed_json))
import json
print(json.dumps(parsed_json, indent=2))

<class 'dict'>
{
  "country": "India",
  "capital": "New Delhi"
}
